# 04 逻辑回归 Logistic Regression

依赖安装说明：`pip install numpy matplotlib scikit-learn`

逻辑回归用于分类，尤其是二分类。虽然名字里有“回归”，但它输出的是类别概率。


## 0. 学习目标和阅读地图

逻辑回归是很多分类模型的起点。学完后你应该能解释：

1. 为什么线性打分 `z` 要经过 sigmoid。
2. 交叉熵为什么比 MSE 更适合分类概率。
3. 阈值变化如何影响 precision 和 recall。
4. 逻辑回归为什么仍然是线性决策边界。


## 1. 数学逻辑

先做一个线性打分：

$$z = w^Tx + b$$

再用 sigmoid 把打分压到 0 到 1：

$$p(y=1|x)=\sigma(z)=\frac{1}{1+e^{-z}}$$

二分类交叉熵损失是：

$$L = -\frac{1}{n}\sum_i[y_i\log(p_i)+(1-y_i)\log(1-p_i)]$$

如果 `p >= 0.5`，通常预测为 1，否则预测为 0。


## 1.1 推导拆开看：logit、odds 和概率

逻辑回归不是直接拟合 `y`，而是拟合 log odds：

$$\log\frac{p}{1-p}=w^Tx+b$$

把它反解回来，就是 sigmoid：

$$p=\frac{1}{1+e^{-(w^Tx+b)}}$$

如果 `z=0`，概率是 0.5；如果 `z` 很大，概率接近 1；如果 `z` 很小，概率接近 0。

交叉熵来自最大似然估计。对于正确类别，它会奖励高概率、惩罚低概率：

$$L_i=-\log p_{correct}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_classification(n_samples=250, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k')
plt.title('二分类数据')
plt.show()


## 1.2 数据流和决策边界

这里 `X` 是二维，所以可以画出决策面。逻辑回归先算：

$$z=w_1x_1+w_2x_2+b$$

当 `z=0` 时，`p=0.5`，这条线就是分类边界。二维里是直线，高维里是超平面。


In [ ]:
# 从零实现：sigmoid + BCE + 梯度下降

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(X_train_s.shape[1])
b = 0.0
lr = 0.1
loss_history = []

for step in range(400):
    z = X_train_s @ w + b
    p = sigmoid(z)
    loss = -np.mean(y_train * np.log(p + 1e-12) + (1 - y_train) * np.log(1 - p + 1e-12))
    loss_history.append(loss)

    grad_z = p - y_train
    grad_w = X_train_s.T @ grad_z / len(X_train_s)
    grad_b = np.mean(grad_z)
    w -= lr * grad_w
    b -= lr * grad_b

print('从零训练 w:', np.round(w, 3))
print('从零训练 b:', round(b, 3))
print('最终 loss:', round(loss_history[-1], 3))

plt.plot(loss_history)
plt.title('逻辑回归训练 loss')
plt.xlabel('step')
plt.ylabel('BCE')
plt.show()


## 1.3 从零实现代码怎么读

训练循环里的关键是 `grad_z = p - y_train`。它说明：

- 如果真实是 1，但模型概率 `p` 太小，梯度会推动 `z` 变大。
- 如果真实是 0，但模型概率 `p` 太大，梯度会推动 `z` 变小。

这和线性回归的残差很像，只是分类场景下残差发生在概率空间。


In [ ]:
model = LogisticRegression()
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
prob = model.predict_proba(X_test_s)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Precision:', round(precision_score(y_test, pred), 3))
print('Recall:', round(recall_score(y_test, pred), 3))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred))

xx, yy = np.meshgrid(np.linspace(X_train_s[:, 0].min()-1, X_train_s[:, 0].max()+1, 160),
                     np.linspace(X_train_s[:, 1].min()-1, X_train_s[:, 1].max()+1, 160))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=0.35)
plt.scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='coolwarm', edgecolor='k')
plt.title('逻辑回归的概率决策面')
plt.show()


In [ ]:
# 诊断：改变阈值会改变 precision 和 recall
thresholds = np.linspace(0.1, 0.9, 17)
precisions, recalls = [], []
for th in thresholds:
    pred_th = (prob >= th).astype(int)
    precisions.append(precision_score(y_test, pred_th, zero_division=0))
    recalls.append(recall_score(y_test, pred_th, zero_division=0))

plt.plot(thresholds, precisions, marker='o', label='precision')
plt.plot(thresholds, recalls, marker='o', label='recall')
plt.title('分类阈值对 precision / recall 的影响')
plt.xlabel('threshold')
plt.ylabel('score')
plt.legend()
plt.show()


## 2.1 如何诊断逻辑回归

分类模型不能只看 accuracy。尤其在类别不平衡时，模型全部预测多数类也可能有很高 accuracy。

常见看法：

- 更关心少漏报：降低阈值，提高 recall。
- 更关心少误报：提高阈值，提高 precision。
- 需要综合平衡：看 F1、PR 曲线或 ROC-AUC。


## 2. 常见误区

- 逻辑回归的决策边界默认是线性的；不是所有分类问题都适合。
- 类别严重不平衡时，accuracy 可能误导，要看 precision、recall、F1。
- 输出概率需要校准时，要额外做 calibration，不要默认它总是可靠概率。

## 3. 小实验

- 改 `class_sep`，观察分类难度变化。
- 把阈值从 `0.5` 改成 `0.3`，观察 recall 和 precision。
- 加多项式特征，让线性模型获得非线性边界。


## 5. 复习清单

- 逻辑回归输出概率，但决策边界默认是线性的。
- sigmoid 把任意实数变成 0 到 1 的概率。
- 交叉熵惩罚“给正确类别低概率”。
- 阈值不是固定必须 0.5，业务目标决定阈值。
